## (un)weighted / (un)directed + non-uniform

In [10]:
import igraph as ig
import numpy as np
import warnings

# allh mia

In [14]:
# apparently i need this
def safe_xlogx(x):
    return np.where(x > 0, x * np.log2(x), 0.0)

In [15]:
# with non-uniform: d=out_strength / total_out_strength
def pagerank_nonuniform(M, tau=0.15, tol=1e-15, maxiter=int(1e6)):
    N = M.shape[0]
    row_sums = M.sum(axis=1)  # out-strength
    dangling = (row_sums == 0)
    row_sums_safe = np.where(dangling, 1, row_sums)
    M_norm = M / row_sums_safe[:, None]  # row-stochastic

    # non-uniform teleportation distribution
    total_out = row_sums.sum()
    d = row_sums / total_out if total_out > 0 else np.ones(N) / N

    p = np.ones(N) / N
    for _ in range(maxiter):
        dangling_sum = p[dangling].sum()
        p_new = (1 - tau) * (p @ M_norm + dangling_sum * d) + tau * d
        if np.linalg.norm(p_new - p) < tol:
            return p_new
        p = p_new

    warnings.warn("did not converge")
    return p

In [16]:
# exit flow για directed: proportional to p[i] * w_ij / out_strength[i]
def compute_exit_flow_nonuniform(g, communities, p):
    communities = np.array(communities)
    out_strength = np.array(g.strength(mode="out", weights="weight" if g.is_weighted() else None), dtype=float)
    weights = np.array(g.es["weight"] if g.is_weighted() else [1.0] * g.ecount())
    edges = np.array(g.get_edgelist(), dtype=int)

    src, trg = edges[:, 0], edges[:, 1]
    src_com, trg_com = communities[src], communities[trg]
    betw = src_com != trg_com

    # avoid div by zero
    out_str_safe = np.where(out_strength > 0, out_strength, 1.0)
    flow = p[src] * weights / out_str_safe[src]

    exit_flow = np.zeros(max(communities) + 1)
    np.add.at(exit_flow, src_com[betw], flow[betw])
    return exit_flow

In [17]:
# exit weights για undirected
def compute_exit_weights(g, communities):
    communities = np.array(communities)
    weights = np.array(g.es["weight"] if g.is_weighted() else [1.0] * g.ecount())
    edges = np.array(g.get_edgelist(), dtype=int)

    src_com = communities[edges[:, 0]]
    trg_com = communities[edges[:, 1]]
    betw = src_com != trg_com

    exit_weights = np.zeros(max(communities) + 1)
    np.add.at(exit_weights, src_com[betw], weights[betw])
    np.add.at(exit_weights, trg_com[betw], weights[betw])
    return exit_weights

In [18]:
# map equation για (un)directed + (un)weighted με non-uniform teleportation (directed only)
def compute_description_length(g, communities, tau=0.15):
    communities = np.array(communities)
    num_comms = max(communities) + 1
    N = g.vcount()

    if g.is_directed():
        adj = np.array(g.get_adjacency(attribute="weight" if g.is_weighted() else None).data, dtype=float)
        p = pagerank_nonuniform(adj, tau=tau)

        p_mod = np.zeros(num_comms)
        np.add.at(p_mod, communities, p)

        exit_flow = compute_exit_flow_nonuniform(g, communities, p)

        # non-uniform teleportation: d per node, d_mod per community
        out_strength = adj.sum(axis=1)
        total_out = out_strength.sum()
        d = out_strength / total_out if total_out > 0 else np.ones(N) / N

        d_mod = np.zeros(num_comms)
        np.add.at(d_mod, communities, d)

        # exit prob: teleportation out of community + edge flow out of community
        q_mod = tau * (1 - d_mod) * p_mod + (1 - tau) * exit_flow

    else:
        weights = np.array(g.es["weight"] if g.is_weighted() else [1.0] * g.ecount())
        total_weight_x2 = 2 * np.sum(weights)

        strength = np.array(g.strength(weights="weight" if g.is_weighted() else None), dtype=float)
        p = strength / total_weight_x2

        p_mod = np.zeros(num_comms)
        np.add.at(p_mod, communities, p)

        exit_weights = compute_exit_weights(g, communities)
        q_mod = exit_weights / total_weight_x2

    q_sum = np.sum(q_mod)
    p_loop = p_mod + q_mod

    L = safe_xlogx(q_sum) - 2 * np.sum(safe_xlogx(q_mod)) \
        - np.sum(safe_xlogx(p)) + np.sum(safe_xlogx(p_loop))

    return L

# testing on Laura's igraphs

In [22]:
# unweighted + undirected
g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/unweighted_undirected.graphml")
communities = g.community_infomap()
L_custom = compute_description_length(g, communities.membership)
L_igraph = communities.codelength
print("--- unweighted + undirected ---")
print("Custom L:", L_custom)
print("igraph L:", L_igraph)

unweighted + undirected:
Custom L: 3.4015829735877707
igraph L: 3.40158297358777


In [23]:
# weighted + undirected
g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/weighted_undirected.graphml")
communities = g.community_infomap(edge_weights="weight")
L_custom = compute_description_length(g, communities.membership)
L_igraph = communities.codelength
print("--- weighted + undirected ---")
print("Custom L:", L_custom)
print("igraph L:", L_igraph)

weighted + undirected:
Custom L: 3.3367124727600905
igraph L: 3.336712472760091


In [24]:
# unweighted + directed
g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/unweighted_directed.graphml")
communities = g.community_infomap()
L_custom = compute_description_length(g, communities.membership)
L_igraph = communities.codelength
print("--- unweighted + directed ")
print("Custom L:", L_custom)
print("igraph L:", L_igraph)

unweighted + directed
Custom L: 3.795838963196576
igraph L: 3.5158296117409074


In [25]:
# weighted + directed
g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/weighted_directed.graphml")
communities = g.community_infomap(edge_weights="weight")
L_custom = compute_description_length(g, communities.membership)
L_igraph = communities.codelength
print("weighted + directed:")
print("Custom L:", L_custom)
print("igraph L:", L_igraph)

weighted + directed:
Custom L: 3.5353257575218677
igraph L: 3.392047186399967
